# analyze_v4_csc2_candidates.ipynb

Downstream analysis for the **6 v4 sources with a CSC2 match within 3σ_pos**.

Reads `outputs/v4_crossmatch_table.csv` (produced by
`scripts/build_v4_crossmatch_table.py`) and filters to
`has_csc2_match_within_3sigma == True`.

## Scope

This notebook does **not** verify any Chandra counterpart association. It:

1. Tabulates per-source SPT-anchor vs CSC2-anchor catalog matches
2. Surfaces cases where the SPT-anchor IR/radio match disagrees with the
   CSC2-anchor result (likely chance alignment in crowded Galactic plane)
3. Decodes AllWISE per-band quality flags
4. Aggregates multi-wavelength fluxes for downstream SED work

## What is still missing for verified association

- `csc2_covered` (footprint coverage) — placeholder `Unknown`, pending
  CSC2 stack-level footprint API
- `csc2_chance_alignment_p` — placeholder `NaN`, needs local CSC2 source
  density estimate
Figures here are for looking. The batch PNG versions come from
`scripts/plot_csc2_cutout_unwise.py`; all figure logic lives in
`src/source_figure.py` and `src/wide_zoom_figure.py`.


In [ ]:
import os, sys
REPO = os.path.dirname(os.path.abspath('.'))   # notebooks/ -> repo root
sys.path.insert(0, os.path.join(REPO, 'src'))
from paths import OUT

# ── Load + filter to CSC2-matched subset ──
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

df_all = pd.read_csv(os.path.join(OUT, 'v4_crossmatch_table.csv'))
df = df_all[df_all['has_csc2_match_within_3sigma']].reset_index(drop=True)

print(f'Loaded {len(df_all)} rows from v4_crossmatch_table.csv')
print(f'Filtered to {len(df)} CSC2-matched sources within 3σ_pos_heuristic')
print()
print('Source list:')
cols = ['id', 'snr_max', 'abs_b_deg', 'sigma_pos_arcsec_heuristic',
        'CSC2_sep_arcsec', 'CSC2_flux_erg_cm2_s', 'CSC2_r0', 'CSC2_r1',
        'simbad_name', 'simbad_otype']
print(df[cols].to_string(index=False))


In [ ]:
# ── AllWISE quality flag decode (per band) ──
# qph = photometric quality, ccf = contamination/confusion; meanings in src/allwise_flags.py
from allwise_flags import QPH_MEANING, CCF_MEANING, decode

print('Per-source AllWISE quality flags (W1 / W2 / W3 / W4):\n')
for _, r in df.iterrows():
    w1, w2 = r.get('AllWISE_flux_mag'), r.get('AllWISE_W2mag')
    w1w2 = (w1 - w2) if (pd.notna(w1) and pd.notna(w2)) else None
    print(r['id'])
    print(f'  W1={w1} mag, W2={w2} mag, W1-W2={w1w2:.2f}' if w1w2 is not None
          else f'  W1={w1} mag, W2={w2} mag')
    for b, ch, meaning in decode(r.get('AllWISE_qph'), QPH_MEANING):
        print(f'    {b} qph={ch}  → {meaning}')
    for b, ch, meaning in decode(r.get('AllWISE_ccf'), CCF_MEANING):
        print(f'    {b} ccf={ch}  → {meaning}')
    # Stern+2012 caveat
    if w1w2 is not None:
        cls = 'AGN candidate' if w1w2 >= 0.8 else 'not AGN by W1-W2 cut'
        print(f'  Stern+2012 (W1-W2≥0.8): {cls}')
        print(f'  CAVEAT: Stern+2012 uses Vega mags + WISE source photometry + depth cuts.')
        print(f'          Our values are AllWISE catalog; needs full quality cuts before use.')
    print()


In [ ]:
# ── Multi-wavelength flux summary (SED building block) ──
# Column names carry units. Radio displayed in mJy (Jy cols ×1000).
RADIO = {
    'AT20G'   : ('AT20G_flux_Jy',     1000.0, 20.0),
    'NVSS'    : ('NVSS_flux_mJy',     1.0,    1.4),
    'RACS-mid': ('RACS-mid_flux_mJy', 1.0,    1.367),
    'VLASS'   : ('VLASS_flux_Jy',     1000.0, 3.0),
}

print('Radio fluxes (mJy):\n')
hdr = f"  {'id':30s} " + ' '.join(f'{c:>10s}' for c in RADIO)
print(hdr); print('  ' + '-'*(len(hdr)-2))
for _, r in df.iterrows():
    vals = []
    for cat, (col, scale, nu) in RADIO.items():
        v = r.get(col)
        vals.append(f'{v*scale:>10.1f}' if pd.notna(v) else f'{"—":>10s}')
    print(f"  {r['id']:30s} " + ' '.join(vals))

print('\nAllWISE W1..W4 (mag) + W1-W2:\n')
for _, r in df.iterrows():
    w1, w2 = r.get('AllWISE_flux_mag'), r.get('AllWISE_W2mag')
    w3, w4 = r.get('AllWISE_W3mag'), r.get('AllWISE_W4mag')
    print(f"  {r['id']:30s} W1={w1!s:>7} W2={w2!s:>7} W3={w3!s:>7} "
          f"W4={w4!s:>7}  W1-W2={r.get('W1_W2')!s}")

print('\nCSC2 broad-band 0.5-7 keV flux (erg/cm2/s):\n')
for _, r in df.iterrows():
    print(f"  {r['id']:30s} Fluxb={r.get('CSC2_flux_erg_cm2_s')!s:>12}  "
          f"r0={r.get('CSC2_r0')!s}\" r1={r.get('CSC2_r1')!s}\"")


In [ ]:
# ── Save subset CSV ──
KEEP_PREFIX = ('id','snr_max','abs_b_deg','l_deg','b_deg',
               'sigma_pos_arcsec_heuristic',
               'simbad_','CSC2_','AT20G_','NVSS_','RACS-mid_','VLASS_','AllWISE_',
               'has_csc2_match_within_3sigma','csc2_')
keep_cols = [c for c in df.columns if c.startswith(KEEP_PREFIX)]
out = df[keep_cols].copy()
out_path = os.path.join(OUT, 'v4_csc2_candidates_summary.csv')
out.to_csv(out_path, index=False)
print(f'Saved {len(out)} rows × {len(out.columns)} cols → {out_path}')

# Literature cross-references for the 6 sources
print('\n=== Literature cross-references ===')
known = {
    'SPT3G_J173508.3-293001.9': (
        'Wan+2025 (arXiv 2509.08962) Source 2: SPT-SV J173508.3-292956. '
        'Position 0.07″ offset from our v4 centroid — likely same physical source.'),
}
for _, r in df.iterrows():
    sid = r['id']
    if sid in known:
        print(f'  ✓ {sid}: {known[sid]}')
    elif pd.notna(r.get('simbad_name')):
        otype = r.get('simbad_otype', '?')
        print(f'  · {sid}: SIMBAD={r["simbad_name"]} ({otype})')
    else:
        print(f'  · {sid}: no SIMBAD match within 30″')


In [ ]:
# ── Per-source cutout panels ──
# unWISE W1/W2 + AllWISE W3/W4 | DECaPS r | RACS-mid, SPT 90/150 GHz contours,
# SPT centroid + σ_pos circle, CSC2 position + 95% circle.
# Reads data/fits_cache/ (run scripts/precache_fits.py first).
# Same figures as PNGs: python scripts/plot_csc2_cutout_unwise.py
import matplotlib.pyplot as plt
from source_figure import csc2_marker, plot_source, spt_markers

for _, r in df.iterrows():
    markers = spt_markers(r) + [m for m in [csc2_marker(r)] if m]
    fig, failures = plot_source(r, markers=markers)
    plt.show()
    for key, err in failures.items():
        print(f'  [{key}] {err[:90]}')


In [ ]:
# ── Wide (3′ DECaPS r) + zoom (30″ DECaPS g/r/i) ──
# HiPS-based, morphology only (no colorbar) — see src/wide_zoom_figure.py
from wide_zoom_figure import plot_wide_zoom

OUT_WAN = os.path.join(OUT, 'images', 'v4_csc2_candidates_wan_style')
os.makedirs(OUT_WAN, exist_ok=True)

for idx, (_, r) in enumerate(df.iterrows()):
    csc2 = (float(r['CSC2_cat_ra']), float(r['CSC2_cat_dec']),
            max(float(r['CSC2_r0']), float(r['CSC2_r1'])))
    fig, failures = plot_wide_zoom(r, csc2=csc2, title_prefix=f'[{idx+1}/{len(df)}] ')
    fpath = os.path.join(OUT_WAN, f"{idx+1:02d}_{r['id']}.png")
    fig.savefig(fpath, dpi=150, bbox_inches='tight')
    print(f'Saved {fpath}', failures or '')
    plt.show()
